In [2]:
import numpy as np
import pandas as pd
import mne
import math
import os

In [3]:
# loading windows
base_dir = r"C:\Users\jshin\OW_closedloopLIFU\eeg_windows\scope\active"
theta_df = []
csv_files = [f for f in os.listdir(base_dir) if f.endswith(".csv")]
csv_files.sort(key=lambda f: os.path.getctime(os.path.join(base_dir, f)))

for idx, window in enumerate(csv_files):
    path = os.path.join(base_dir, csv_files[idx])
    idx_df = pd.read_csv(path) 
    theta_df.append(idx_df)
theta_df[1]

,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel,offline z-score,median,mad,LIFU,t_rel,idx
0,1.002504e+06,38029.031250,-8.813601,77.679565,48.086876,10.949013,10.430403,-0.211308,NaN,NaN,NaN,0.0,-1.995294,1
1,1.002504e+06,37762.144531,-8.356216,69.826355,48.707367,11.097456,10.430403,-0.219347,NaN,NaN,NaN,0.0,-1.985266,1
2,1.002504e+06,38238.718750,-7.784257,60.594658,48.997803,11.166937,10.430403,-0.129410,NaN,NaN,NaN,0.0,-1.984481,1
3,1.002504e+06,38575.675781,-7.116444,50.643768,48.977459,11.162071,10.430403,0.067644,NaN,NaN,NaN,0.0,-1.982924,1
4,1.002504e+06,38143.949219,-6.372707,40.611401,48.690449,11.093410,10.430403,0.310418,NaN,NaN,NaN,0.0,-1.974126,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2243,1.002513e+06,37943.488281,4.702597,22.114420,8.986072,1.594754,1.338846,-0.845066,1.338846,-0.137662,1.338846,0.0,6.977381,1
2244,1.002513e+06,38422.925781,5.398940,29.148554,9.554222,1.730675,1.338846,-0.804079,1.338846,-0.137662,1.338846,0.0,6.981064,1
2245,1.002513e+06,38121.917969,6.009997,36.120068,10.254125,1.898116,1.338846,-0.678651,1.338846,-0.137662,1.338846,0.0,6.983394,1
2246,1.002513e+06,37561.300781,6.521281,42.527100,11.072462,2.093890,1.338846,-0.502642,1.338846,-0.137662,1.338846,0.0,6.989660,1


In [5]:
#when it sonicates
lifu_markers = pd.read_csv("lifu_markers_1_2back_active.csv")
lifu_on_df = lifu_markers[lifu_markers["marker"] == "LIFU_ON"]
lifu_on = np.array(lifu_on_df["LSL_timestamp"])
lifu_on

array([1002279.2847846, 1002505.9063606])

In [6]:
#finding baseline median and mean
medians = []
mads = []
tol = 0.002

for idx, event in enumerate(lifu_on):
    df = theta_df[idx]
    diff = np.abs(df["LSL Time"].values - event)
    i = np.argmin(diff)
    row = df.iloc[i]
    median = row["median"]
    mad = row["mad"]
    medians.append(median)
    mads.append(mad)
print(f"medians = {medians}")
print(f"mads = {mads}")

medians = [0.0780445337295532, -0.0671638436615466]
mads = [1.5047277212142944, 1.5659065246582031]


In [7]:
# median percent change calculation
idx = 1
pre_sonication= []
post_sonication= []
percent_change_df = pd.DataFrame(lifu_on, columns=["Time"])

#for loop
for idx, event in enumerate(lifu_on):
    df = theta_df[idx]
    pre_df = df[df["LSL Time"]<= event]
    pre_mean = np.mean(np.array(pre_df["Hold"].dropna()))
    post_df = df[df["LSL Time"]>= event]
    post_mean = np.mean(np.array(post_df["Hold"].dropna()))
    pre_sonication.append(pre_mean)
    post_sonication.append(post_mean)
percent_change_df["pre average median"] = pre_sonication
percent_change_df["post average median"] = post_sonication
# CHECK IF EQUATION IS CORRECT -- also slope
percent_change_df["percent change %"] = 100* (percent_change_df["post average median"] - percent_change_df["pre average median"])/ percent_change_df["pre average median"]
percent_change_df

,Time,pre average median,post average median,percent change %
0,1.002279e+06,1.251860,1.18412,-5.411173
1,1.002506e+06,8.184259,-0.06274,-100.766597


In [8]:
df = theta_df[1].copy()
time = lifu_on[1]
df = df[df['LSL Time'] > time]
median = df.iloc[np.argmin(np.abs(df["LSL Time"] - time))]['median']
# for each row, compare to median (median is more robust than mean)
threshold = 0.5 # do actual calculation based on mad
df = df[df['median'] < median +threshold] #<= threshold]
df


,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel,offline z-score,median,mad,LIFU,t_rel,idx
499,1.002506e+06,38446.574219,-3.700580,13.694290,8.902260,1.574703,1.565907,0.758015,1.565907,-0.067164,1.565907,0.0,0.000270,1
500,1.002506e+06,37889.960938,-3.688895,13.607946,8.907033,1.575845,1.565907,0.382031,1.565907,-0.067164,1.565907,0.0,0.004502,1
501,1.002506e+06,37703.398438,-3.618439,13.093104,8.879552,1.569271,1.565907,-0.013588,1.565907,-0.067164,1.565907,0.0,0.018223,1
502,1.002506e+06,38223.242188,-3.491190,12.188406,8.822246,1.555561,1.565907,-0.266208,1.565907,-0.067164,1.565907,0.0,0.018888,1
503,1.002506e+06,38488.199219,-3.310011,10.956172,8.739950,1.535873,1.565907,-0.332243,1.565907,-0.067164,1.565907,0.0,0.019205,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2243,1.002513e+06,37943.488281,4.702597,22.114420,8.986072,1.594754,1.338846,-0.845066,1.338846,-0.137662,1.338846,0.0,6.977381,1
2244,1.002513e+06,38422.925781,5.398940,29.148554,9.554222,1.730675,1.338846,-0.804079,1.338846,-0.137662,1.338846,0.0,6.981064,1
2245,1.002513e+06,38121.917969,6.009997,36.120068,10.254125,1.898116,1.338846,-0.678651,1.338846,-0.137662,1.338846,0.0,6.983394,1
2246,1.002513e+06,37561.300781,6.521281,42.527100,11.072462,2.093890,1.338846,-0.502642,1.338846,-0.137662,1.338846,0.0,6.989660,1


In [9]:
df_sorted = df.sort_values("LSL Time", ascending=False) # descending
MIN_SPACING = 1.0  # seconds
kept_rows = []
last_time = 0

for _, row in df_sorted.iterrows():
    t = row["LSL Time"]
    if np.abs(t - last_time) >= MIN_SPACING:
        kept_rows.append(row)
        last_time = t
    if len(kept_rows) == 15:
        break
drift_time = pd.DataFrame(kept_rows)
drift_time = np.array(drift_time['LSL Time'])
drift_time = np.sort(drift_time)
drift_time

array([1002506.8850754, 1002507.8861288, 1002508.8872973, 1002509.8917662,
       1002510.8947437, 1002511.9038928, 1002512.9058703])